# Day C6 — spike train 분석: raster, PSTH, 발화율, ISI, tuning curve

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/C-neural-signals/day-C6.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/C-neural-signals/day-C6.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — Poisson 스파이크 만들기, raster 행렬, PSTH, SDF

In [ ]:
import numpy as np

rng = np.random.default_rng(9)
t0, t1 = -0.5, 1.0                     # 자극 전 0.5초 ~ 후 1.0초
def rate(t):                            # 참 발화율(Hz): 기저 5 Hz, 자극 후 0.1–0.3초에 40 Hz까지 오름
    return 5 + 35 * np.exp(-((t - 0.2) ** 2) / (2 * 0.05 ** 2))

def inhomogeneous_poisson(rate_fn, r_max):
    """thinning: 최대 발화율로 균일 Poisson 스파이크를 만든 뒤, rate/r_max 확률로 남긴다"""
    n = rng.poisson(r_max * (t1 - t0))
    cand = np.sort(rng.uniform(t0, t1, n))
    return cand[rng.uniform(0, 1, n) < rate_fn(cand) / r_max]

n_tr = 60
trains = [inhomogeneous_poisson(rate, 40) for _ in range(n_tr)]      # 시행마다 스파이크 시각 목록
print("시행 1의 스파이크 시각(초):", np.round(trains[0], 3))
print("시행당 평균 스파이크 수: %.1f" % np.mean([len(s) for s in trains]))

# raster를 0/1 행렬로: 1 ms 칸
dt = 0.001
edges = np.arange(t0, t1 + dt / 2, dt)
raster = np.array([np.histogram(s, edges)[0] for s in trains])        # 시행 × 시간
print("raster 행렬 모양:", raster.shape, "| 칸 하나에 스파이크 2개 이상인 곳:", int((raster > 1).sum()))

# PSTH: 20 ms 칸, 단위를 '스파이크/초(Hz)'로
bw = 0.020
pedges = np.arange(t0, t1 + bw / 2, bw)
counts = np.sum([np.histogram(s, pedges)[0] for s in trains], axis=0)
psth = counts / n_tr / bw
centers = pedges[:-1] + bw / 2
for c in [-0.25, 0.11, 0.21, 0.61]:
    i = np.argmin(np.abs(centers - c))
    print("t = %+.2f초: PSTH %.1f Hz (참값 %.1f Hz)" % (centers[i], psth[i], rate(centers[i])))

# spike density function: 1 ms raster의 시행 평균을 가우시안 커널(σ = 15 ms)로 매끄럽게
sig = 0.015
kt = np.arange(-4 * sig, 4 * sig + dt / 2, dt)
kern = np.exp(-kt ** 2 / (2 * sig ** 2)); kern /= kern.sum()
sdf = np.convolve(raster.mean(axis=0) / dt, kern, mode="same")
tc = edges[:-1] + dt / 2
print("SDF 최고점: %.3f초, %.1f Hz" % (tc[np.argmax(sdf)], sdf.max()))

### 예시 2 — ISI, CV, Fano factor, 그리고 불응기

In [ ]:
import numpy as np

rng = np.random.default_rng(10)
def poisson_train(r, T, dead=0.0):
    """일정 발화율 r Hz의 Poisson 스파이크. dead > 0이면 스파이크 뒤 dead초 동안 못 쏜다(불응기)"""
    isi = rng.exponential(1 / r, int(r * T * 3)) + dead
    st = np.cumsum(isi)
    return st[st < T]

def describe(name, trains, T):
    isi = np.concatenate([np.diff(s) for s in trains])
    cv = isi.std() / isi.mean()                                 # ISI 변동계수
    counts = np.array([len(s) for s in trains])
    fano = counts.var(ddof=1) / counts.mean()                   # 시행 간 스파이크 수의 분산/평균
    short = np.mean(isi < 0.002) * 100
    print("%-18s 발화율 %.1f Hz | CV %.2f | Fano %.2f | ISI<2ms %.1f%%" % (name, counts.mean() / T, cv, fano, short))

T, r = 2.0, 20
describe("Poisson", [poisson_train(r, T) for _ in range(200)], T)
describe("Poisson+불응기 3ms", [poisson_train(r / (1 - r * 0.003), T, 0.003) for _ in range(200)], T)
# 규칙적으로 쏘는 세포: ISI가 평균 50 ms, SD 5 ms인 정규분포
def regular_train(T):
    st = np.cumsum(np.abs(rng.normal(0.05, 0.005, 80)))
    return st[st < T]
describe("규칙적 발화", [regular_train(T) for _ in range(200)], T)

# 자기상관(autocorrelogram)의 핵심: 모든 스파이크 쌍의 시차를 모은다 (±30 ms)
s = poisson_train(r / (1 - r * 0.003), 200, 0.003)
d = s[None, :] - s[:, None]
lags = np.abs(d[(d != 0) & (np.abs(d) <= 0.03)])
print("시차 0–2 ms인 쌍: %d개 | 시차 10–12 ms인 쌍: %d개" % ((lags < 0.002).sum(), ((lags > 0.010) & (lags < 0.012)).sum()))

### 예시 3 — 방향 tuning curve와 cosine 적합

In [ ]:
import numpy as np
from scipy.optimize import curve_fit, minimize

rng = np.random.default_rng(11)
dirs = np.deg2rad(np.arange(0, 360, 45))            # 8방향
n_rep = 12                                          # 방향마다 12번 반복
pref_true = np.deg2rad(135)
def cos_tune(th, base, depth, pref):
    return base + depth * np.cos(th - pref)

theta = np.repeat(dirs, n_rep)
T = 0.5                                             # 0.5초 창에서 센 스파이크 수
lam = np.clip(cos_tune(theta, 20, 15, pref_true), 0, None) * T
counts = rng.poisson(lam)
rate = counts / T                                   # 스파이크/초

mean_r = np.array([rate[theta == d].mean() for d in dirs])
sem_r = np.array([rate[theta == d].std(ddof=1) / np.sqrt(n_rep) for d in dirs])
for d, m, s in zip(np.rad2deg(dirs), mean_r, sem_r):
    print("%3d°: %5.1f ± %.1f Hz" % (d, m, s))

# (1) 최소제곱 cosine 적합 (시행 하나하나를 넣는다)
p, cov = curve_fit(cos_tune, theta, rate, p0=[rate.mean(), 5, 0])
base, depth, pref = p
if depth < 0:                                       # 진폭이 음수로 나오면 같은 곡선을 180° 돌려 표현
    depth, pref = -depth, pref + np.pi
print("최소제곱: 기저 %.1f Hz, 변조 %.1f Hz, 선호 방향 %.0f°" % (base, depth, np.rad2deg(pref) % 360))

# (2) Poisson 우도 최대화: log(λ) = b0 + b1 cosθ + b2 sinθ  (Poisson GLM과 같은 모양)
X = np.column_stack([np.ones_like(theta), np.cos(theta), np.sin(theta)])
def nll(b):
    eta = X @ b
    return np.sum(np.exp(eta) * T - counts * eta)   # 음의 로그우도 (상수항 생략)
b = minimize(nll, np.zeros(3)).x
print("Poisson GLM: 선호 방향 %.0f° (참값 135°)" % (np.rad2deg(np.arctan2(b[2], b[1])) % 360))